# Fine-Tune SmolLM2-1.7B for Football Tactical Analysis

Uses **LlamaFactory** (same approach as `ocr_finetune_vlm.ipynb`).

- **Method**: LoRA via LlamaFactory CLI + QLoRA 4-bit
- **Dataset**: `data/finetune/football_tactical.jsonl` (ShareGPT chat format)
- **GPU**: GTX 1660 Ti 6GB or Colab T4

---
### To switch to a different model later
Only change `MODEL_NAME_OR_PATH` in **Section 3** and `template` in **Section 4**. Everything else is reusable.

## 1. Installation

In [ ]:
# Same versions as ocr_finetune_vlm.ipynb reference
!pip install transformers==4.57.6
!pip install optimum==1.26.0
!pip install datasets==4.4.0
!pip install peft
!pip install torch==2.8.0
!pip install torchvision==0.23
!pip install torchaudio==2.8.0

# Clone LlamaFactory (pinned to same commit as reference)
!git clone --depth 1 https://github.com/hiyouga/LlamaFactory.git
!cd LlamaFactory && git checkout 762b480131908d37736ad9aa3f12e87f8f7e6313

!cd LlamaFactory && pip install -e .
!cd LlamaFactory && pip install -r requirements/metrics.txt

## 2. Prepare Dataset — Convert JSONL → ShareGPT JSON

LlamaFactory expects a **JSON array** in ShareGPT format:
```json
[{"conversations": [{"from": "human", "value": "..."}, {"from": "gpt", "value": "..."}]}, ...]
```
Our `football_tactical.jsonl` already has `messages` with `role/content` â€” we just convert.

In [1]:
import json, os, random

# ── Paths ────────────────────────────────────────────────────────────────────

JSONL_PATH  = "../data/finetune/football_tactical.jsonl"   # source JSONL
TRAIN_JSON  = "../data/finetune/football_train.json"                        # output train
VAL_JSON    = "../data/finetune/football_val.json"                          # output val
VAL_SPLIT   = 0.20                                         # 20% validation
SEED        = 42

# ── Load JSONL ───────────────────────────────────────────────────────────────

records = []
with open(JSONL_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        messages = rec.get("messages", [])

        # Convert role/content â†’ from/value  (ShareGPT format)
        role_map = {"user": "human", "assistant": "gpt", "system": "system"}
        conversations = [
            {"from": role_map.get(m["role"], m["role"]), "value": m["content"]}
            for m in messages
        ]
        
        # For SmolLM2, we optionally want a unified system prompt early in the conversation if missing
        if len(conversations) > 0 and conversations[0]["from"] != "system":
            system_msg = {
                "from": "system",
                "value": (
                    "You are an expert football tactical analyst. "
                    "Analyse the match data provided, predict the most likely outcome, "
                    "and deliver a detailed tactical report covering both teams' "
                    "strengths, weaknesses, and key strategic factors."
                )
            }
            conversations.insert(0, system_msg)

        records.append({"conversations": conversations})

print(f"Total records: {len(records)}")

# ── Train / Val split ────────────────────────────────────────────────────────
random.Random(SEED).shuffle(records)
val_n       = max(1, int(len(records) * VAL_SPLIT))
val_ds      = records[:val_n]
train_ds    = records[val_n:]

with open(TRAIN_JSON, "w", encoding="utf-8") as f:
    json.dump(train_ds, f, ensure_ascii=False)
with open(VAL_JSON, "w", encoding="utf-8") as f:
    json.dump(val_ds,   f, ensure_ascii=False)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")
print(f"\nSample conversation:")
print(json.dumps(train_ds[0]["conversations"][0], indent=2))

Total records: 1752
Train: 1402 | Val: 350

Sample conversation:
{
  "from": "system",
  "value": "You are an expert football tactical analyst. Analyse the match data provided, predict the most likely outcome, and deliver a detailed tactical report covering both teams' strengths, weaknesses, and key strategic factors."
}


## 3. Register Dataset in LlamaFactory

Add entries to `LlamaFactory/data/dataset_info.json` so the CLI can find our files.

In [ ]:
import json, os

TRAIN_ABS = os.path.abspath(TRAIN_JSON)
VAL_ABS   = os.path.abspath(VAL_JSON)

info_path = "LlamaFactory/data/dataset_info.json"
with open(info_path, encoding="utf-8") as f:
    dataset_info = json.load(f)

# ── Add our two splits ────────────────────────────────────────────────────────
dataset_info["football_train"] = {
    "file_name": TRAIN_ABS,
    "formatting": "sharegpt",
    "columns": {"messages": "conversations"}
}
dataset_info["football_val"] = {
    "file_name": VAL_ABS,
    "formatting": "sharegpt",
    "columns": {"messages": "conversations"}
}

with open(info_path, "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, indent=2, ensure_ascii=False)

print(f"✓ Registered 'football_train' and 'football_val'")
print(f"  Train file: {TRAIN_ABS}")
print(f"  Val   file: {VAL_ABS}")

âœ“ Registered 'football_train' and 'football_val'
  Train file: /football_train.json
  Val   file: /football_val.json


## 4. Write Training YAML Config

The YAML controls the model, LoRA settings, dataset, and training hyper-parameters.

> **To add a new model**: change `model_name_or_path` and `template`. That is all.

In [ ]:
import os

# ── ⚙️  CHANGE ONLY THIS BLOCK TO SWITCH MODELS ──────────────────────────────
MODEL_NAME_OR_PATH = "HuggingFaceTB/SmolLM2-1.7B-Instruct"   # HuggingFace model id
TEMPLATE           = "chatml"                           # LlamaFactory template name (SmolLM2 uses ChatML)
OUTPUT_DIR         = "../models/finetuned_smollm2"     # where checkpoints go

# MODEL_NAME_OR_PATH = "Qwen/Qwen2.5-1.5B-Instruct"   # HuggingFace model id
# TEMPLATE           = "qwen"                           # LlamaFactory template name
# OUTPUT_DIR         = "../models/finetuned_qwen25"     # where checkpoints go
 # ─────────────────────────────────────────────────────────────────────────────


os.makedirs(OUTPUT_DIR, exist_ok=True)
YAML_PATH = "LlamaFactory/examples/train_lora/football_smollm2.yaml"
os.makedirs(os.path.dirname(YAML_PATH), exist_ok=True)

yaml_content = f"""
### model
model_name_or_path: {MODEL_NAME_OR_PATH}
use_fast_tokenizer: false
cache_dir: ../models/cache
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_alpha: 32
# lora_dropout: 0.05
lora_target: all
use_rslora: true

### dataset
dataset: football_train
eval_dataset: football_val
template: {TEMPLATE}
cutoff_len: 1024
overwrite_cache: true
preprocessing_num_workers: 2

### output
# resume_from_checkpoint: {os.path.abspath(OUTPUT_DIR)}/checkpoint-100
output_dir: {os.path.abspath(OUTPUT_DIR)}
logging_steps: 10
save_steps: 50
save_total_limit: 3
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 2.0e-4
num_train_epochs: 10
lr_scheduler_type: cosine
warmup_ratio: 0.05
fp16: true
ddp_timeout: 180000000

### eval
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 50

report_to: wandb
run_name: yt-football-finetune-llamafactory
""".strip()

with open(YAML_PATH, "w") as f:
    f.write(yaml_content)

print(f"✓ YAML written to: {YAML_PATH}")
print()
print(yaml_content)


âœ“ YAML written to: LlamaFactory/examples/train_lora/football_smollm2.yaml

### model
model_name_or_path: HuggingFaceTB/SmolLM2-1.7B-Instruct
use_fast_tokenizer: false
cache_dir: ../models/cache
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_alpha: 32
# lora_dropout: 0.05
lora_target: all
use_rslora: true

### dataset
dataset: football_train
eval_dataset: football_val
template: chatml
cutoff_len: 1024
overwrite_cache: true
preprocessing_num_workers: 2

### output
# resume_from_checkpoint: /models/finetuned_smollm2/checkpoint-100
output_dir: /models/finetuned_smollm2
logging_steps: 10
save_steps: 50
save_total_limit: 3
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 2.0e-4
num_train_epochs: 10
lr_scheduler_type: cosine
warmup_ratio: 0.05
fp16: true
ddp_timeout: 180000000

### eval
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 50

re

## 5. Login to HuggingFace (SmolLM2-1.7B is public â€” only needed for gated models)

In [ ]:
# Uncomment if running on Colab with userdata
# from google.colab import userdata
# hf_token = userdata.get('huggingface')
# !huggingface-cli login --token {hf_token}

# Or paste your token directly:
# !huggingface-cli login --token hf_YOUR_TOKEN_HERE

print("SmolLM2-1.7B-Instruct is public â€” no token needed.")
print("Uncomment the lines above if you use a private/gated model.")

## 6. Train

In [4]:
import os
# !pip install wandb
os.environ['WANDB_API_KEY'] = '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'
print('WandB API key set successfully via environment variable.')


WandB API key set successfully via environment variable.


In [9]:
import subprocess, os

yaml_abs = os.path.abspath(YAML_PATH)

os.environ["DISABLE_VERSION_CHECK"] = "1"
!cd LlamaFactory && llamafactory-cli train {yaml_abs}

[WARNING|2026-03-09 15:26:17] llamafactory.extras.misc:155 >> Version checking has been disabled, may lead to unexpected behaviors.
/usr/local/lib/python3.12/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[INFO|2026-03-09 15:26:18] llamafactory.hparams.parser:508 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16
[INFO|tokenization_utils_base.py:2111] 2026-03-09 15:26:18,845 >> loading file vocab.json from cache at ../models/cache/models--HuggingFaceTB--SmolLM2-1.7B-Instruct/snapshots/31b70e2e869a7173562077fd711b654946d38674/vocab.json
[INFO|tokenization_utils_base.py:2111] 2026-03-09 15:26:18,845 >> loading file merges.txt from cache at ../models/cache/models--HuggingFaceTB--Smol

In [19]:
LORA_PATH = os.path.abspath(OUTPUT_DIR)
LORA_PATH

'/models/finetuned_smollm2'

In [ ]:
!huggingface-cli login --token ""

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

# Upload the entire directory
api.upload_folder(
    folder_path="./LlamaFactory/saves/smollm_sft_merged",
    repo_id="aliosama8399/football-finetuning",  # Replace with your repo
    repo_type="model",  # or "dataset" or "space"
    # path_in_repo="/custom/folder/"
)

## 7. Inference with Fine-Tuned Model

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

LORA_PATH = r"C:\Users\Ali\.unsloth\studio\exports\unsloth_qwen3-0.6b_1774335162\checkpoint-360"
# HF_MODEL_ID = "aliosama8399/football-finetuning"
# Load base model + LoRA adapter
tokenizer = AutoTokenizer.from_pretrained(LORA_PATH, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    LORA_PATH,
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
# model = PeftModel.from_pretrained(base_model, LORA_PATH)
base_model.eval()

def generate_analysis(prompt: str, max_new_tokens: int = 512) -> str:
    """
    Generate tactical analysis from fine-tuned SmolLM2.
    """
    messages = [
        {"role": "system", "content": "You are an expert football tactical analyst. Analyse the match data provided, predict the most likely outcome, and deliver a detailed tactical report covering both teams' strengths, weaknesses, and key strategic factors."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(base_model.device)
    with torch.no_grad():
        out = base_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


# ── Test ──────────────────────────────────────────────────────────────────────

test_prompt = """Analyze the upcoming match: Manchester City (Home) vs Arsenal (Away).

"""

print("=" * 70)
result = generate_analysis(test_prompt)
print(result)

The tokenizer you are loading from 'C:\Users\Ali\.unsloth\studio\exports\unsloth_qwen3-0.6b_1774335162\checkpoint-360' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
`torch_dtype` is deprecated! Use `dtype` instead!


<think>
Okay, so I need to analyze this Manchester City vs Arsenal match. Let me start by recalling the key points of each team's recent form and recent matches. Manchester City has been solid, showing excellent defensive cohesion and attacking quality, while Arsenal has been quite effective at defending but is often a vulnerable spot on the road. I should check recent fixtures to see if such a dynamic clash between two consistently powerful sides is likely.

First, Manchester City's home advantage is a significant factor. They tend to be more comfortable in their own half, and their defensive structure is formidable. They have been excellent at absorbing pressure, with a well-oiled engine that can control games. Their main weakness, however, is their recent defensive line. While they are not entirely defenseless, they are not impervious and can be breached. Their forwards are also a key point, showing incredible attacking potential and skill to create chances.

Arsenal's strengths in 

In [1]:
# Load model directly
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("aliosama8399/football-analysisN", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained("aliosama8399/football-analysisN" ,dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True)

model.eval()
def generate_analysis(prompt: str, max_new_tokens: int = 2048) -> str:
    """
    Generate tactical analysis from fine-tuned Qwen2.5.
    To swap models: just change MODEL_NAME_OR_PATH and LORA_PATH above.
    """
    messages = [
        {"role": "system", "content": "You are an expert football tactical analyst. Analyze the match, predict the most likely outcome, and deliver a detailed tactical report covering both teams' strengths, weaknesses, and key strategic factors."},
        {"role": "user", "content": prompt}
    ]  
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


# ── Test ──────────────────────────────────────────────────────────────────────
test_prompt = """Analyze the upcoming match: Manchester City (Home) vs Arsenal (Away).

"""

print("=" * 70)
result = generate_analysis(test_prompt)
print(result)

# inputs = tokenizer.apply_chat_template(
# 	messages,
# 	add_generation_prompt=True,
# 	tokenize=True,
# 	return_dict=True,
# 	return_tensors="pt",
# ).to(model.device)

# outputs = model.generate(**inputs, max_new_tokens=40)
# print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))


<think>

</think>

Of course. Here is a comprehensive analysis of the upcoming match between Manchester City and Arsenal, breaking down both teams' strengths and weaknesses while identifying the key tactical battle.

---

### **Match Analysis: Manchester City vs. Arsenal**

### **1. Match Setup: Competitive but Dismissive**

The fixture is defined by a clear hierarchy between the two teams. **Manchester City dominate at the home gate**, thanks to their superior attacking firepower and tactical fluidity. They will likely dominate possession and territory. **Arsenal, while not a dominant force**, will be a more cautious side, relying on defensive resilience and counter-attacking flexibility rather than a fortress like City's. The match will likely end in a high-scoring affair or a tight, scoreless deadlock.

### **2. Team Strengths & Weaknesses**

#### **1. Manchester City (Home)**

*   **Strengths:**
    *   **Dominant Attack:** City's forward attack is a masterpiece. Their combination 

## 8. Integration with `llm_providers.py`

Paste this class into `models/llm_providers.py` and add it to the provider factory:

```python
class FinetunedLlamaFactoryProvider:
    """Provider for any model fine-tuned via LlamaFactory + LoRA."""
    def __init__(self, base_model_id: str, lora_path: str):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        from peft import PeftModel
        import torch
        self.tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
        base = AutoModelForCausalLM.from_pretrained(
            base_model_id, torch_dtype=torch.float16,
            device_map="auto", trust_remote_code=True
        )
        self.model = PeftModel.from_pretrained(base, lora_path).eval()

    def generate_explanation(self, match_context: dict, gnn_explanation: dict) -> str:
        prompt = self._build_prompt(match_context, gnn_explanation)
        messages = [
            {"role": "system", "content": "You are an expert football tactical analyst. Analyse the match data provided, predict the most likely outcome, and deliver a detailed tactical report covering both teams' strengths, weaknesses, and key strategic factors."},
            {"role": "user", "content": prompt}
        ]
        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=512,
                                      temperature=0.7, do_sample=True)
        return self.tokenizer.decode(
            out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
        )
```

Register it:
```python
"smollm2-finetuned": lambda: FinetunedLlamaFactoryProvider(
    base_model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    lora_path     = "models/finetuned_smollm2",
)
```